# ecgaug: run one config on Kaggle

Thin wrapper: it clones the repository at a fixed commit, installs the locked environment, runs one
config through the package CLI, and zips the outputs. All logic lives in `src/ecgaug/`.

Before running:
1. Notebook settings: turn **Internet** on, and pick a GPU if the config uses one.
2. If the repository is private, add a GitHub token with read access as the Kaggle secret `GITHUB_TOKEN`
   (Add-ons, Secrets). The token is never printed.
3. Fill in the parameters in the next cell. `GIT_SHA` must be the full commit you want to run.

Afterwards: download `ecgaug_<sha>_<time>.zip` from the Output tab and run
`python scripts/ingest_runs.py <zip>` in your clone.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
GIT_SHA = ""                           # full 40-character commit SHA (required)
MODULE_CLI = "ecgaug.experiments.m1"   # the package CLI to run
CONFIG = "configs/m1/A.yaml"
EXTRA_ARGS = []                        # e.g. ["--device", "cuda"]
RAW_DATASET_DIR = ""                   # optional: /kaggle/input/<dataset> holding the raw MIT-BIH files

In [ ]:
import datetime, os, shutil, subprocess, sys
from pathlib import Path

assert len(GIT_SHA) == 40, "Set GIT_SHA to the full commit SHA you want to run"
WORK = Path("/kaggle/working")
REPO = WORK / "repo"

env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                       # public repository, or no secret configured
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin", GIT_SHA], cwd=REPO, check=True, env=env)
subprocess.run(["git", "checkout", "--quiet", GIT_SHA], cwd=REPO, check=True)
print("checked out", subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip())

In [ ]:
# Locked environment: exactly the versions in uv.lock.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--no-dev"], cwd=REPO, check=True)
RUN = ["uv", "run", "--frozen", "--no-dev", "python", "-m"]

In [ ]:
# Raw data: copy an attached dataset if given, otherwise download from PhysioNet. Either way every
# file is checked against PhysioNet's published SHA-256 list before anything else runs.
if RAW_DATASET_DIR:
    shutil.copytree(RAW_DATASET_DIR, REPO / "data" / "raw" / "mitdb", dirs_exist_ok=True)
    subprocess.run([*RUN, "ecgaug.data.raw", "--skip-download"], cwd=REPO, check=True)
else:
    subprocess.run([*RUN, "ecgaug.data.raw"], cwd=REPO, check=True)

In [ ]:
# The run itself. The CLI refuses to start if the pre-registration is not in force, the validation
# split is not approved, or the checkout differs from the commit.
subprocess.run([*RUN, MODULE_CLI, "--config", CONFIG, *EXTRA_ARGS], cwd=REPO, check=True)

In [ ]:
# Package runs/ and the variant result files for scripts/ingest_runs.py.
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M")
staging = WORK / "bundle"
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(REPO / "runs", staging / "runs")
if (REPO / "results" / "validation").exists():
    shutil.copytree(REPO / "results" / "validation", staging / "results" / "validation")
bundle = shutil.make_archive(str(WORK / f"ecgaug_{GIT_SHA[:7]}_{stamp}"), "zip", staging)
shutil.rmtree(staging)
print("bundle:", bundle, f"({Path(bundle).stat().st_size / 1e6:.1f} MB)")